# Homework 2 — train a Transformer from scratch

This notebook uses only the supplied `SlavaYus/tinysms_qa_12m` training split. The T5-style encoder–decoder starts with random weights; the included tokenizer is fitted on the same training split. Train on an available Colab GPU (the corporate run uses a G4). The separate inference notebook must be benchmarked on a **T4 GPU** for the course limit. Checkpoints are saved in the Colab runtime; download `latest/` before the runtime ends. The 20,000-update run is an experiment target, not a claim of achieved quality.

In [ ]:
!git clone --depth=1 --branch=2026 https://github.com/OsVox/hse-nlp.git /content/hse-nlp
%cd /content/hse-nlp
!pip -q install 'transformers>=4.53,<5' 'pyarrow>=18' 'requests>=2.31' 

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
assert torch.cuda.is_available(), 'Select a GPU in Runtime → Change runtime type' 

In [ ]:
CHECKPOINT_DIR = '/content/hw2_checkpoints'
print('Checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
!python -m homeworks.hw2.train --steps 20000 --batch-size 16 --accumulation 4 --cache-dir /content/hw2_cache --output-dir "$CHECKPOINT_DIR" --save-every 500 --eval-every 500

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

checkpoint_root = Path(CHECKPOINT_DIR)
checkpoint = checkpoint_root / ('best' if (checkpoint_root / 'best' / 'model.safetensors').exists() else 'latest')
assert (checkpoint / 'model.safetensors').exists(), 'Training checkpoint is missing'
archive_path = Path('/content/hw2_model.zip')
with ZipFile(archive_path, 'w') as archive:
    for path in checkpoint.iterdir():
        if path.is_file() and path.name != 'training_state.pt':
            archive.write(path, arcname=path.name)
print('Exporting checkpoint:', checkpoint)
print('Exported model files:', [path.name for path in checkpoint.iterdir() if path.is_file() and path.name != 'training_state.pt'])
files.download(str(archive_path))

The export cell downloads model and tokenizer files to your computer. `training_state.pt` stays in Colab because it is only needed to resume the optimizer. Keep the training tab open until the ZIP download completes.